In [1]:
# BMI Male/Female Dataset — K-Means Clustering + NLP

## 1. Import Libraries

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
```

## 2. Load the Dataset

Make sure `Bmi_male_female.csv` is in the same folder as your Jupyter Notebook.

```python
df = pd.read_csv("Bmi_male_female.csv")

# Display first 5 rows
df.head()
```

## 3. Explore the Dataset

```python
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns)

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nStatistical summary:")
display(df.describe())
```

## 4. Check the Data

```python
# Display unique values for each column
for column in df.columns:
    print(f"\n{column}:")
    print(df[column].unique()[:20])
```

If the dataset contains columns such as:

* `Gender`
* `Height`
* `Weight`
* `BMI`

we can convert the categorical gender column into numerical form.

```python
df["Gender"] = df["Gender"].map({
    "Male": 0,
    "Female": 1
})

df.head()
```

If your CSV uses `M` and `F` instead:

```python
df["Gender"] = df["Gender"].map({
    "M": 0,
    "F": 1
})
```

## 5. Handle Missing Values

```python
print(df.isnull().sum())

# Remove rows containing missing values
df = df.dropna()

print("Dataset after removing missing values:", df.shape)
```

## 6. Select Features for K-Means

For example, if your dataset contains `Gender`, `Height`, and `Weight`:

```python
features = ["Gender", "Height", "Weight"]

X = df[features]

X.head()
```

If your dataset has only numerical BMI-related measurements, you can instead use:

```python
features = ["Height", "Weight"]

X = df[features]
```

## 7. Standardize the Data

K-Means is distance-based, so scaling is important.

```python
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]
```

## 8. Find the Best Number of Clusters — Elbow Method

```python
inertia = []

K = range(1, 11)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)
```

Plot the elbow curve:

```python
plt.figure(figsize=(8, 5))

plt.plot(K, inertia, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")

plt.grid(True)
plt.show()
```

## 9. Calculate Silhouette Scores

```python
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = kmeans.fit_predict(X_scaled)
    
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))

plt.plot(
    range(2, 11),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.grid(True)
plt.show()
```

## 10. Apply K-Means

Choose the number of clusters based on the elbow plot and silhouette analysis.

For example, using 3 clusters:

```python
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

df.head()
```

## 11. Examine the Clusters

```python
print(df["Cluster"].value_counts())
```

View the average values in each cluster:

```python
cluster_summary = df.groupby("Cluster")[features].mean()

display(cluster_summary)
```

## 12. Visualize the Clusters

If the dataset contains Height and Weight:

```python
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x="Height",
    y="Weight",
    hue="Cluster",
    palette="viridis",
    s=100
)

plt.title("K-Means Clustering of BMI Dataset")
plt.xlabel("Height")
plt.ylabel("Weight")

plt.show()
```

## 13. Visualize Male and Female Distribution

```python
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x="Cluster",
    hue="Gender"
)

plt.title("Gender Distribution Across K-Means Clusters")
plt.xlabel("Cluster")
plt.ylabel("Number of People")

plt.show()
```

## 14. BMI Analysis

If the dataset already contains a BMI column:

```python
if "BMI" in df.columns:
    print(df.groupby("Cluster")["BMI"].mean())
```

Visualize BMI by cluster:

```python
if "BMI" in df.columns:
    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="Cluster",
        y="BMI"
    )

    plt.title("BMI Distribution by Cluster")
    plt.show()
```

## 15. NLP Section

If the dataset contains a text column, such as `Description`, `Comment`, or `Text`, NLP can be applied to that column.

First identify text columns:

```python
text_columns = df.select_dtypes(
    include=["object"]
).columns

print("Text columns:")
print(text_columns)
```

Suppose the text column is called `Description`:

```python
from sklearn.feature_extraction.text import TfidfVectorizer

# Remove missing text
df["Description"] = df["Description"].fillna("")

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=100
)

X_text = vectorizer.fit_transform(df["Description"])

print("TF-IDF matrix shape:", X_text.shape)
```

## 16. K-Means on NLP/TF-IDF Data

```python
text_kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["Text_Cluster"] = text_kmeans.fit_predict(X_text)

df[["Description", "Text_Cluster"]].head()
```

## 17. Display Important Words in Each NLP Cluster

```python
terms = vectorizer.get_feature_names_out()

order_centroids = text_kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster_number in range(3):
    print(f"\nCluster {cluster_number}")
    
    top_words = [
        terms[index]
        for index in order_centroids[cluster_number, :10]
    ]
    
    print(", ".join(top_words))
```

## 18. Save the Results

```python
df.to_csv(
    "BMI_KMeans_Results.csv",
    index=False
)

print("Results saved as BMI_KMeans_Results.csv")
```

## 19. Final Results

```python
display(df.head(20))
```

The resulting dataset will contain an additional `Cluster` column, allowing you to see which K-Means group each male/female record belongs to.


SyntaxError: invalid syntax (1464123851.py, line 5)